# Copying and Understanding the Code

* copy the foundational code examples for structured streaming
* understand the components of PySpark streaming, such as initializing a Spark session, defining schemas, and creating streaming DataFrames

# Step 1: Import required Pyspark libraries from Python and start a SparkSession

In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [49]:
spark  = SparkSession \
    .builder \
    .appName("readfromcsv") \
    .master("local[4]") \
    .getOrCreate()

**Important: Remember that structured streaming processing always requires the specification of a schema for the data in the stream**

In [50]:
schema1 = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("profession", StringType(), True),
    StructField("city", StringType(), True),
    StructField("salary", DoubleType(), True)
]
)

# Step 2: Load data into a streaming DataFrame by using the “readStream” 
*Also check status of streaming with the “isStreaming” method.*

Code Notes:  
* customer = spark.readStream \       # read streaming data from the specified directory
* .format("csv") \                    # specify the format of the input data
* .schema(schema1) \                  # specify the schema of the CSV files
* .option("header", "true") \         # specify that the first line of the CSV files contains the header
* .option("maxFilesPerTrigger", 1) \  # read one file at a time
* .load("/home/jovyan/work/data")     # specify the directory to read the CSV files from/*

In [51]:
customer = spark.readStream \
    .format("csv") \
    .schema(schema1) \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .load("/home/jovyan/work/data")


In [52]:
customer.isStreaming # check if the DataFrame is streaming

True

**Take a look at the schema of DataFrame in a tree format**

In [53]:
customer.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- profession: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)



# Step 3: Apply Transformations
* show # of people from each profession
* descend order in a DF that will update w/ every new file

In [54]:
average_salary = customer.groupBy("profession")\
    .agg((avg("salary").alias("average_salary")),(count("profession").alias("count")))\
    .sort(desc("average_salary"))

**Specify a format() for streaming destination & outputMode() for the determination of the data to be written into a streaming sink**

Options for outputMode() method:
* append: Only new rows will be written to the sink.
* complete: All rows will be written to the sink, every time there are updates.
* update: Only the updated rows will be written to the sink, every time there are updates.
* will also use “complete” option as we have an aggregation in our DataFrame.

**Start streaming using start() method**

In [55]:
query = average_salary.writeStream.format("console").outputMode("complete").start()

*see results in terminal*
* Make sure Docker container is started:
    * </> Bash: docker compose up -d
* See Container Logs
    * </> Bash: docker logs pyspark-jupyter

# Stop Streaming
* use stop() method

In [56]:
query.stop()